<!-- REFACTOR-COMMENT: POST_PROCESSING -->
### Refactor Comment (Post-Processing)
- **Purpose**: Compare subsidy and distortion metrics across scenarios for one subsidies-distortions run.
- **Primary inputs**: `runs/<run_name>/*/subsidies_distortion.csv` and `runs/<run_name>/*/output.csv`.
- **Primary outputs**: `img/subsidies_<select>.csv`, bar charts, `img/subsidies_gap.png`, and per-scenario subsidy-vs-distortion scatters.
- **Refactor role**: Consolidate exploratory cells into a reproducible workflow with explicit configuration and reusable helpers.

<!-- NOTEBOOK-WORKFLOW -->
## Notebook Workflow
### What This Notebook Does
1. Resolve a subsidies-distortions run folder and load all scenario tables.
2. Build `Subsidies Gap = Distortion - Subsidies` for each scenario.
3. Extract selected subsidy indicators from `output.csv`.
4. Export a summary CSV and generate standardized plots.

### Inputs
- `project/analysis/post_processing/policy_assessment/runs/<run_name>/*/subsidies_distortion.csv`
- `project/analysis/post_processing/policy_assessment/runs/<run_name>/*/output.csv`

### Outputs
- `project/analysis/post_processing/policy_assessment/runs/<run_name>/img/subsidies_<select>.csv`
- `project/analysis/post_processing/policy_assessment/runs/<run_name>/img/subsidies_gap.png`
- `project/analysis/post_processing/policy_assessment/runs/<run_name>/img/subsidies_distortion_<scenario>.png`

### How To Reuse
- Update `run_name` and optional filters in the configuration cell.

# Analyze subsidy-distortion runs


In [ ]:
from pathlib import Path

import sys
sys.path.insert(0, "../../../..")

from project.analysis.post_processing.shared.io_runs import load_output_bundle, load_child_csv_bundle
from project.analysis.post_processing.shared.paths import resolve_run
from project.analysis.post_processing.shared.transforms import (
    keep_scenarios,
    add_subsidies_gap,
    build_indicator_summary,
    parse_subsidy_indicator_index,
    subsidy_summary_by_housing_status,
    subsidy_summary_by_income,
    subsidy_delta_relative_to_reference,
    build_ad_valorem_ratio,
)
from project.analysis.post_processing.shared.plots import (
    plot_indicator_bar_grid,
    plot_subsidies_by_housing_status,
    plot_subsidy_delta_by_housing_status,
    plot_subsidy_delta_pct_by_housing_status,
    plot_ad_valorem_ratio_by_housing_status,
    plot_subsidies_gap_boxplot,
    plot_subsidy_vs_distortion,
)
from project.utils import make_scatter_plot

In [ ]:
# Run selection
run_name = "subsidies_distortions"
selected_scenarios = None  # Example: ["subsidies_2018", "subsidies_2021", "subsidies_2024"]

# Indicator extraction
select = "avg"  # "avg" or a year available in output.csv, e.g. 2019
indicator_pattern = r"^Subsidies total (.*) \(Million euro\)$"
investment_pattern = r"^Investment total (.*) \(Billion euro\)$"
reference_pattern = None  # Example: r"^Investment total (.*) \(Billion euro\)$"
reference_scale = 1e3

# Scenario display names used in plots and exported tables
scenario_renames = {"Reference": "Package2024"}

# Reference scenario for relative share plots
optimal_scenario = "OptimalSubsidies"

# Plot controls
same_ylim = True
fontsize = 14

remove_technologies = [
    "Electricity-Direct electric",
    "Natural gas-Performance boiler",
    "Oil fuel-Performance boiler",
    "Wood fuel-Performance boiler",
    "Heating-District heating",
]

In [ ]:
run_folder = resolve_run("policy_assessment", run_name)
output_folder = run_folder / "img"
output_folder.mkdir(parents=True, exist_ok=True)

print(f"Run folder: {run_folder}")
print(f"Output folder: {output_folder}")

distortion_bundle = load_child_csv_bundle(run_folder, "subsidies_distortion.csv", index_col=None)
output_bundle = load_output_bundle(run_folder)

if not distortion_bundle:
    raise FileNotFoundError(f"No subsidies_distortion.csv files found in {run_folder}")
if not output_bundle:
    raise FileNotFoundError(f"No output.csv files found in {run_folder}")

print(f"Loaded distortion tables: {len(distortion_bundle)}")
print(f"Loaded output tables: {len(output_bundle)}")

In [ ]:
distortion_bundle = keep_scenarios(distortion_bundle, selected_scenarios)
output_bundle = keep_scenarios(output_bundle, selected_scenarios)

if selected_scenarios:
    print(f"Filtered scenarios: {list(distortion_bundle.keys())}")

if not distortion_bundle or not output_bundle:
    raise ValueError("Scenario filter removed all data. Update `selected_scenarios`.")

if scenario_renames:
    distortion_bundle = {scenario_renames.get(name, name): table for name, table in distortion_bundle.items()}
    output_bundle = {scenario_renames.get(name, name): table for name, table in output_bundle.items()}
    optimal_scenario = scenario_renames.get(optimal_scenario, optimal_scenario)

distortion_bundle = add_subsidies_gap(distortion_bundle)


In [ ]:
final_df = build_indicator_summary(
    output_bundle=output_bundle,
    pattern=indicator_pattern,
    select_value=select,
    reference=reference_pattern,
    reference_multiplier=reference_scale,
)

if final_df.empty:
    raise ValueError("No indicators matched the configured regex pattern.")

# Wide-format CSV: Metric x Scenario
summary_path = output_folder / f"subsidies_{select}.csv"
final_df.to_csv(summary_path, index=True)
print(f"Saved wide-format summary: {summary_path}")

final_df

In [ ]:
# Subsidy levels by income class — current vs. optimal policy package
parsed = parse_subsidy_indicator_index(final_df)
scenarios = list(final_df.columns)

save_path = output_folder / f"subsidies_{select}_by_income_and_segment.png"
plot_subsidies_by_housing_status(
    parsed, scenarios=scenarios, label_size=fontsize, same_axis_limits=same_ylim, save=save_path,
)
print(f"Saved: {save_path}")

# Wide-format CSV with parsed structure
parsed_wide = parsed[parsed["Income"] != "Total"].pivot_table(
    index=["Housing", "Status", "Income"], columns="Scenario", values="Value",
)
parsed_wide_path = output_folder / f"subsidies_{select}_by_segment_income.csv"
parsed_wide.to_csv(parsed_wide_path)
print(f"Saved: {parsed_wide_path}")

In [ ]:
# Misallocation of subsidies vs. optimal — by income class and housing segment
delta_df = subsidy_delta_relative_to_reference(parsed, reference_scenario=optimal_scenario)

for scenario in [s for s in scenarios if s != optimal_scenario]:
    # Absolute difference (Million euro)
    save_eur = output_folder / f"subsidies_delta_{scenario}_vs_{optimal_scenario}.png"
    plot_subsidy_delta_by_housing_status(
        delta_df, scenario=scenario, optimal_scenario=optimal_scenario,
        label_size=fontsize, same_axis_limits=same_ylim, save=save_eur,
    )
    print(f"Saved: {save_eur}")

    # Relative difference (%)
    save_pct = output_folder / f"subsidies_delta_pct_{scenario}_vs_{optimal_scenario}.png"
    plot_subsidy_delta_pct_by_housing_status(
        delta_df, scenario=scenario, optimal_scenario=optimal_scenario,
        label_size=fontsize, same_axis_limits=same_ylim, save=save_pct,
    )
    print(f"Saved: {save_pct}")

In [ ]:
# Implicit ad valorem rate: subsidies / investment by housing segment
ad_valorem_df = build_ad_valorem_ratio(
    output_bundle,
    subsidy_pattern=indicator_pattern,
    investment_pattern=investment_pattern,
    select_value=select,
)

av_path = output_folder / f"ad_valorem_ratio_{select}.csv"
(ad_valorem_df * 100).to_csv(av_path)
print(f"Saved: {av_path}")

save_path = output_folder / f"ad_valorem_ratio_{select}.png"
plot_ad_valorem_ratio_by_housing_status(
    ad_valorem_df, label_size=fontsize, same_axis_limits=same_ylim, save=save_path,
)
print(f"Saved: {save_path}")

## Aggregated views and policy gap analysis

In [ ]:
# Total subsidies by housing segment
hs_df = subsidy_summary_by_housing_status(parsed)
hs_path = output_folder / f"subsidies_{select}_by_housing_status.csv"
hs_df.to_csv(hs_path)
print(f"Saved: {hs_path}")

plot_indicator_bar_grid(
    hs_df,
    same_axis_limits=same_ylim,
    label_size=fontsize,
    value_formatter=lambda value: f"{value:,.0f} M€",
)

In [ ]:
# Total subsidies by income class
inc_df = subsidy_summary_by_income(parsed)
inc_path = output_folder / f"subsidies_{select}_by_income.csv"
inc_df.to_csv(inc_path)
print(f"Saved: {inc_path}")

plot_indicator_bar_grid(
    inc_df,
    same_axis_limits=same_ylim,
    label_size=fontsize,
    value_formatter=lambda value: f"{value:,.0f} M€",
)

In [ ]:
plot_subsidies_gap_boxplot(distortion_bundle, output_folder / "subsidies_gap.png")


In [ ]:
plot_subsidy_vs_distortion(
    bundle=distortion_bundle,
    output_folder=output_folder,
    technology_filter=remove_technologies,
    make_scatter_fn=make_scatter_plot,
)